# CAT-Loss War Room

> **DEMO RESEARCH MEMO** - This notebook is for demonstration purposes only.
> It is **not legal advice**. All outputs must be independently verified.
> **Verify all citations** (KeyCite / Shepardize) before any legal reliance.

---

AI-powered catastrophic loss litigation research. Given a case intake, this notebook
generates targeted research queries across weather data, carrier intel, and case law.

In [ ]:
"""Cell 1: Imports + Configuration"""
from war_room.notebook_runtime import ensure_runtime_context

context = ensure_runtime_context(globals())

print("==================================================")
print("WAR ROOM CONFIG")
print("==================================================")
print(f"  APP_ENV:            {context.settings.app_env.value}")
print(f"  USE_CACHE:          {context.settings.use_cache}")
print(f"  CACHE_DIR:          {context.settings.cache_dir}")
print(f"  CACHE_SAMPLES_DIR:  {context.settings.cache_samples_dir}")
print(f"  OUTPUT_DIR:         {context.settings.output_dir}")
print(f"  LIVE_RETRIEVAL:     {context.settings.live_retrieval_enabled}")
print(f"  EXA_API_KEY:        {'***set***' if context.settings.exa_api_key_value else 'not set'}")
print("==================================================")


In [ ]:
"""Cell 2: Scenario Selection + Case Intake"""
from war_room.notebook_runtime import prepare_notebook_scenario

SCENARIO_ID = "milton_pinellas_citizens_ho3"
SCENARIO_OVERRIDES = {
    # "coverage_issues": ["wind vs water causation", "scope of repair"],
    # "posture": ["denial", "underpayment"],
}

selection = prepare_notebook_scenario(
    SCENARIO_ID,
    overrides=SCENARIO_OVERRIDES,
    namespace=globals(),
)
scenario = selection.scenario
intake = selection.intake
case_key = selection.case_key

print("Available scenarios:")
for item in selection.available_scenarios:
    default_marker = " (selected)" if item.slug == selection.selected_slug else ""
    offline_marker = " [offline-ready]" if item.offline_demo_ready else ""
    print(f"  - {item.slug}{default_marker}{offline_marker}: {item.title}")

print()
print(f"Selected scenario: {scenario.title}")
print(f"Benchmark focus: {scenario.benchmark_focus}")
if scenario.notes:
    print(f"Notes: {scenario.notes[0]}")
if selection.warning_message:
    print(selection.warning_message)
print()
print(intake.format_card())


In [ ]:
"""Cell 3: Query Plan Generation"""
from war_room.query_plan import build_research_plan, format_query_plan
from war_room.workflow_summary import format_research_plan_preview

research_plan = build_research_plan(intake)
queries = research_plan.query_plan
print(format_research_plan_preview(research_plan))
print()
print(format_query_plan(queries))


In [ ]:
"""Cell 4: Weather Corroboration"""
from war_room.exa_client import ExaClient
from war_room.notebook_runtime import ensure_runtime_context
from war_room.weather_module import build_weather_brief

context = ensure_runtime_context(globals())

client = None
if context.settings.live_retrieval_enabled and context.settings.exa_api_key_value:
    try:
        client = ExaClient(api_key=context.settings.exa_api_key_value)
    except Exception:
        print("Unable to initialize Exa client - running from cache only")
else:
    print("Live retrieval disabled or EXA_API_KEY missing - running from cache only")

weather = build_weather_brief(
    intake,
    client,
    query_plan=queries,
    use_cache=context.settings.use_cache,
    cache_dir=str(context.settings.cache_dir),
    cache_samples_dir=str(context.settings.cache_samples_dir),
)

print(f"Event: {weather['event_summary']}")
print(f"Sources: {len(weather['sources'])}")
print(f"Metrics: {weather['metrics']}")
print()
print("Key Observations:")
for i, obs in enumerate(weather['key_observations'][:5], 1):
    print(f"  {i}. {obs[:150]}")
print()
print("Top Sources:")
for s in weather['sources'][:5]:
    print(f"  {s['badge']} {s['title'][:60]}")
    print(f"    {s['url']}")


In [ ]:
"""Cell 5: Carrier Document Pack"""
from war_room.carrier_module import build_carrier_doc_pack
from war_room.notebook_runtime import ensure_runtime_context

context = ensure_runtime_context(globals())

carrier = build_carrier_doc_pack(
    intake,
    client,
    query_plan=queries,
    use_cache=context.settings.use_cache,
    cache_dir=str(context.settings.cache_dir),
    cache_samples_dir=str(context.settings.cache_samples_dir),
)

print(f"Carrier: {carrier['carrier_snapshot']['name']}")
print(f"Documents: {len(carrier['document_pack'])}")
print()
print("Document Pack:")
for d in carrier['document_pack'][:8]:
    print(f"  {d['badge']} [{d['doc_type']}] {d['title'][:60]}")
print()
print("Common Defenses:")
for d in carrier['common_defenses']:
    print(f"  - {d}")
print()
print("Rebuttal Angles:")
for r in carrier['rebuttal_angles']:
    print(f"  -> {r}")


In [ ]:
"""Cell 6: Case Law + Citation Spot-Check"""
from IPython.display import HTML, display
from war_room.caselaw_module import build_caselaw_pack
from war_room.citation_verify import spot_check_citations
from war_room.evidence_board import build_evidence_board_from_parts, format_evidence_board, render_evidence_board_html
from war_room.issue_workspace import build_issue_workspace_from_parts, format_issue_workspace
from war_room.notebook_runtime import ensure_runtime_context

context = ensure_runtime_context(globals())

caselaw = build_caselaw_pack(
    intake,
    client,
    query_plan=queries,
    use_cache=context.settings.use_cache,
    cache_dir=str(context.settings.cache_dir),
    cache_samples_dir=str(context.settings.cache_samples_dir),
)

print("Case Law by Issue:")
for issue in caselaw['issues']:
    print(f"\n  [{issue['issue']}]")
    for c in issue['cases']:
        cite = f" -- {c['citation']}" if c.get('citation') else ""
        print(f"    {c['badge']} {c['name'][:50]}{cite}")

print("\n" + "=" * 50)
print("Citation Spot-Check")
print("=" * 50)

citecheck = spot_check_citations(
    caselaw,
    client,
    use_cache=context.settings.use_cache,
    cache_dir=str(context.settings.cache_dir),
    cache_samples_dir=str(context.settings.cache_samples_dir),
)

for c in citecheck['checks']:
    print(f"  {c['badge']} {c['case_name'][:40]} -- {c['note'][:60]}")

s = citecheck['summary']
print(f"\nSummary: {s['verified']} verified, {s['uncertain']} uncertain, {s['not_found']} not found")
print(f"\n{citecheck['disclaimer']}")

evidence_board = build_evidence_board_from_parts(
    intake,
    weather,
    carrier,
    caselaw,
    citecheck,
    queries,
)

print()
try:
    display(HTML(render_evidence_board_html(evidence_board)))
except Exception:
    print(format_evidence_board(evidence_board))

issue_workspace = build_issue_workspace_from_parts(
    intake,
    weather,
    carrier,
    caselaw,
    citecheck,
    queries,
)

print()
print(format_issue_workspace(issue_workspace))


In [ ]:
"""Cell 7: Export Research Memo"""
from war_room.export_history import build_export_history_from_parts, format_export_history
from war_room.export_md import render_markdown_memo, write_markdown
from war_room.memo_composer import build_memo_composer_from_parts, format_memo_composer
from war_room.notebook_runtime import ensure_runtime_context
from war_room.workflow_summary import build_run_timeline, format_run_timeline

context = ensure_runtime_context(globals())

memo_md = render_markdown_memo(
    intake,
    weather,
    carrier,
    caselaw,
    citecheck,
    queries,
)

memo_composer = build_memo_composer_from_parts(
    intake,
    weather,
    carrier,
    caselaw,
    citecheck,
    queries,
)

output_path = write_markdown(str(context.settings.output_dir), case_key, memo_md)
run_record, run_stages = build_run_timeline(
    intake,
    research_plan,
    weather,
    carrier,
    caselaw,
    citecheck,
    environment="notebook",
    export_written=True,
)

export_history = build_export_history_from_parts(
    intake,
    weather,
    carrier,
    caselaw,
    citecheck,
    queries,
    run_status=run_record.status,
    export_written=True,
    artifact_uri=output_path,
)

print(format_memo_composer(memo_composer))
print()
print(format_run_timeline(run_record, run_stages))
print()
print(format_export_history(export_history))
print()
print(f"Memo saved to: {output_path}")
print(f"Length: {len(memo_md)} characters")
print()
for line in memo_md.split('\n')[:40]:
    print(line)
